# ML Zoomcamp - Homework 3: Classification

Flujo completo siguiendo la metodología del curso:
- Limpieza y preparación de datos
- Correlaciones lineales y moda
- Partición 60/20/20 con semilla fija
- Cálculo de Mutual Information
- Entrenamiento de Regresión Logística con DictVectorizer
- Feature Elimination para sensibilidad de atributos
- Regularización C e inspección de pesos

In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

## 1. Carga y Normalización

In [15]:
data_url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

try:
    df = pd.read_csv(data_url)
except Exception:
    df = pd.read_csv('leads.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

string_columns = list(df.dtypes[df.dtypes == 'object'].index)
for col in string_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### Q1: Moda de 'industry'

In [16]:
mode_val = df['industry'].mode()[0]
print(df['industry'].value_counts(dropna=False))
print(f'\nRESPUESTA Q1: {mode_val}')

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NaN               240
Name: count, dtype: int64

RESPUESTA Q1: technology


### Q2: Correlación numérica

In [17]:
target_col = 'converted'
numerical_cols = list(df.select_dtypes(include=['int64', 'float64']).columns)
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

categorical_cols = list(df.select_dtypes(include=['object', 'string']).columns)
corr_matrix = df[numerical_cols].corr()
corr_pairs = corr_matrix.abs().unstack()
corr_pairs = corr_pairs[corr_pairs < 0.9999].sort_values(ascending=False)

print(corr_matrix.round(3))
print(f'\nRESPUESTA Q2: {corr_pairs.index[0]} con valor {corr_pairs.iloc[0]:.4f}')

                          annual_income  number_of_courses_viewed  \
annual_income                     1.000                     0.138   
number_of_courses_viewed          0.138                     1.000   
interaction_count                 0.106                     0.722   
lead_score                        0.231                     0.769   

                          interaction_count  lead_score  
annual_income                         0.106       0.231  
number_of_courses_viewed              0.722       0.769  
interaction_count                     1.000       0.929  
lead_score                            0.929       1.000  

RESPUESTA Q2: ('interaction_count', 'lead_score') con valor 0.9287


## 2. Marco de Validación (60/20/20)

In [18]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

del df_train[target_col]
del df_val[target_col]
del df_test[target_col]

### Q3: Mutual Information

In [19]:
def calculate_mi(series):
    return mutual_info_score(series.fillna('missing'), y_train)

mi_scores = df_train[categorical_cols].apply(calculate_mi).sort_values(ascending=False)
print(mi_scores.round(2))
print(f'\nRESPUESTA Q3: {mi_scores.index[0]} ({mi_scores.iloc[0]:.2f})')

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

RESPUESTA Q3: lead_source (0.03)


### Q4: Regresión Logística y Exactitud

In [20]:
all_features = categorical_cols + numerical_cols

df_train_prepared = df_train[all_features].fillna(0)
df_val_prepared = df_val[all_features].fillna(0)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train_prepared.to_dict(orient='records'))
X_val = dv.transform(df_val_prepared.to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

original_accuracy = accuracy_score(y_val, model.predict(X_val))
print(f'RESPUESTA Q4: {original_accuracy:.2f} (exacto: {original_accuracy:.4f})')

RESPUESTA Q4: 0.65 (exacto: 0.6450)


### Q5: Feature Elimination

In [21]:
diffs = []
for feat in all_features:
    sub_cols = [f for f in all_features if f != feat]
    dv_sub = DictVectorizer(sparse=False)
    X_tr = dv_sub.fit_transform(df_train_prepared[sub_cols].to_dict(orient='records'))
    X_va = dv_sub.transform(df_val_prepared[sub_cols].to_dict(orient='records'))
    
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    acc = accuracy_score(y_val, m.predict(X_va))
    diff = original_accuracy - acc
    diffs.append({'feature': feat, 'diff': diff, 'abs_diff': abs(diff)})

df_diffs = pd.DataFrame(diffs).sort_values(by='abs_diff')
print(df_diffs)
print(f'\nRESPUESTA Q5: {df_diffs.iloc[0]["feature"]}')

                    feature   diff  abs_diff
1                  industry  0.000     0.000
2         employment_status  0.000     0.000
3                  location  0.000     0.000
7                lead_score  0.001     0.001
5  number_of_courses_viewed  0.002     0.002
0               lead_source  0.003     0.003
6         interaction_count  0.044     0.044
4             annual_income -0.080     0.080

RESPUESTA Q5: industry


### Q6: Regularización C

In [23]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    m = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(f'C={c:<5} -> Accuracy: {round(acc, 3)} (exacto: {acc:.4f})')

C=1e-06 -> Accuracy: 0.598 (exacto: 0.5980)
C=1e-05 -> Accuracy: 0.598 (exacto: 0.5980)
C=0.0001 -> Accuracy: 0.613 (exacto: 0.6130)
C=0.001 -> Accuracy: 0.645 (exacto: 0.6450)
